# 08 · Thesis figures (layout of the student's previous version)

Reproduces the student's figure set with the V3 results: the spatial validation layout, the final model comparison, the confusion matrix with the mean ROC curve, and the SHAP and permutation-importance panels. **Nothing is refitted:** every figure reads the outputs of notebooks 03–05 and 07 (outer-fold predictions, fold metrics, the frozen final model's name, permutation importance and SHAP values).

| Figure | Content | Folder |
|---|---|---|
| Fig. A | Spatial validation layout: blockCV blocks coloured by fold; purge circles for one outer fold | `restricted` (borehole locations) |
| Fig. B | Final model comparison: balanced accuracy, mean over the 5 outer spatial test folds | `aggregate` |
| Fig. C | Confusion matrix (pooled outer test folds) and mean spatial ROC curve of the final model | `aggregate` |
| Fig. D | Global SHAP importance, held-out permutation importance and SHAP beeswarm of the final model | `aggregate` |

Settings in the next cell: `CELLS = "all"` (measured + filled test cells, as in the CV) or `"measured"`; `FOLD` for the purge panel.

In [ ]:
import os, sys, json
from pathlib import Path

# Run mode: "full" for the experiment, "smoke" for the reduced software validation run.
# os.environ["SIEMREAP_MODE"] = "smoke"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "project.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from siemreap import config
cfg = config.load_config(ROOT)
MODE = config.run_mode()
print(config.smoke_banner())

import geopandas as gpd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from siemreap import folds as F, geo
AGG = config.stage_dir(cfg, "08_thesis_figures", "aggregate")
RES = config.stage_dir(cfg, "08_thesis_figures", "restricted")
preds = pd.read_csv(config.stage_dir(cfg, "03_nested_cv", "restricted") / "outer_predictions.csv")
summ = pd.read_csv(config.stage_dir(cfg, "04_results", "aggregate") / "metrics_fold_mean_sd.csv")
final_model = json.loads((config.stage_dir(cfg, "05_final", "aggregate") / "final_model.json").read_text())["model"]
primary = int(preds.block_m.mode()[0]) if MODE == "smoke" else F.primary_block(cfg)
mt = pd.read_csv(config.processed_dir(cfg) / "cells_all.csv")

CELLS = "all"          # "all" (measured + filled test cells) or "measured"
FOLD = 1               # outer fold shown in the purge panel
if CELLS == "measured":
    preds = preds[preds.measured == 1]
    from siemreap import metrics as M
    summ = M.fold_summary(M.fold_table(preds))
main = preds[(preds.block_m == primary) & (preds.purge_m == 0)]
print("final model:", final_model, "| primary block:", primary, "m | cells:", CELLS)

# palette of the previous version
MAGENTA, WINE, TEAL, YELLOW, NAVY, GREY = "#A0004E", "#6E0032", "#17918C", "#E5B74A", "#17365C", "#9E9E9E"
SEQ = [MAGENTA, WINE, TEAL, YELLOW, NAVY]
FOLD_COL = ["#2C6FBB", "#F28E2B", "#59A14F", "#C0000B", "#7B5CA8"]
CMAP = LinearSegmentedColormap.from_list("navy_magenta", [NAVY, "#D9A6C1", MAGENTA])
NICE = {"RF": "Random Forest", "XGB": "XGBoost", "MLP": "MLP", "GWRF": "GWRF", "GWXGB": "GWXGBoost",
        "DEPTH_PRIOR": "Depth only\n(reference)"}
plt.rcParams.update({"font.family": "sans-serif", "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": False, "figure.facecolor": "white", "savefig.dpi": 300, "savefig.bbox": "tight"})

def save(fig, folder, name):
    for ext in ("png", "pdf"):
        fig.savefig(folder / f"{name}.{ext}")
    print("saved", folder / f"{name}.png")

## Fig. A · Spatial validation layout

In [ ]:
folds = F.load_outer(cfg, primary)
blk = gpd.read_file(config.stage_dir(cfg, "04_blockcv_folds", "restricted") / f"blocks_{primary}m.gpkg")
bh = mt[mt.in_A].drop_duplicates("BH_Code").set_index("BH_Code")[["UTM_Easting_m", "UTM_Northing_m"]]
bh["fold"] = folds.fold.reindex(bh.index).astype(int)
pts = gpd.GeoDataFrame(bh, geometry=gpd.points_from_xy(bh.UTM_Easting_m, bh.UTM_Northing_m), crs=cfg["data"]["crs_epsg"])
occupied = blk[blk.intersects(pts.union_all())]
km = lambda v, _: f"{v / 1000:.0f}"
fig, axes = plt.subplots(1, 2, figsize=(18, 7.2))
for ax in axes:
    for k in sorted(bh.fold.unique()):
        occupied[occupied.folds == k].plot(ax=ax, color=FOLD_COL[k - 1], alpha=0.22, edgecolor=FOLD_COL[k - 1], linewidth=0.6)
        b = bh[bh.fold == k]
        ax.scatter(b.UTM_Easting_m, b.UTM_Northing_m, s=42, color=FOLD_COL[k - 1], edgecolor="white", linewidth=0.8,
                   label=f"Fold {k}", zorder=3)
    ax.xaxis.set_major_formatter(km); ax.yaxis.set_major_formatter(km)
    ax.set_xlabel("UTM Easting (km)", fontsize=12); ax.set_ylabel("UTM Northing (km)", fontsize=12)
    ax.set_aspect("equal")
    for s in ax.spines.values():
        s.set_visible(True); s.set_color("#666666")
axes[0].set_title(f"Development Spatial Validation Layout\n", fontsize=16, fontweight="bold")
axes[0].text(0.5, 1.01, f"{primary} m spatial blocks (blockCV, k = 5)", transform=axes[0].transAxes, ha="center", fontsize=11, color="#555555")
axes[0].legend(frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0))
axes[0].text(0.01, 0.01, f"Transparent squares = occupied {primary} m spatial blocks", transform=axes[0].transAxes, fontsize=8.5, color="#555555")
# purge panel: training boreholes within the purge distance of a fold-FOLD test borehole are removed
test = bh[bh.fold == FOLD]; train = bh[bh.fold != FOLD]
d = geo.nearest_distance(train[["UTM_Easting_m", "UTM_Northing_m"]].values, test[["UTM_Easting_m", "UTM_Northing_m"]].values)
removed = train[d < primary]
for x, y in test[["UTM_Easting_m", "UTM_Northing_m"]].values:
    axes[1].add_patch(plt.Circle((x, y), primary, fill=False, color="#888888", linewidth=0.9, zorder=2))
axes[1].scatter(removed.UTM_Easting_m, removed.UTM_Northing_m, s=70, marker="x", color="black", linewidth=1.4, zorder=4)
axes[1].set_title(f"Spatial Purge Layout (outer fold {FOLD})\n", fontsize=16, fontweight="bold")
axes[1].text(0.5, 1.01, f"{primary} m blocks | training boreholes within {primary} m of a fold-{FOLD} test borehole removed",
             transform=axes[1].transAxes, ha="center", fontsize=11, color="#555555")
h, l = axes[0].get_legend_handles_labels()
axes[1].legend(h + [Line2D([], [], color="#888888", lw=0.9), Line2D([], [], marker="x", ls="", color="black")],
               l + [f"{primary} m purge circle (fold {FOLD} test)", f"removed training borehole ({len(removed)})"],
               frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0))
fig.tight_layout()
save(fig, RES, "figA_spatial_validation_layout")

## Fig. B · Final model comparison

In [ ]:
SHOW_SD = True          # error bars = SD over the 5 outer folds
s = summ[(summ.block_m == primary) & (summ.purge_m == 0)].sort_values("balanced_accuracy_mean", ascending=False)
ml = s[s.model != "DEPTH_PRIOR"]
order = list(ml.model) + ["DEPTH_PRIOR"]
s = s.set_index("model").loc[order]
cols = [SEQ[i % len(SEQ)] for i in range(len(ml))] + [GREY]
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(order))
sd = s.balanced_accuracy_sd.fillna(0)
ax.bar(x, s.balanced_accuracy_mean, color=cols, width=0.82,
       yerr=sd if SHOW_SD else None, capsize=4, error_kw={"elinewidth": 1, "ecolor": "#444444"})
for xi, v in zip(x, s.balanced_accuracy_mean):
    ax.text(xi, v + (sd.max() if SHOW_SD else 0) + 0.01, f"{v:.4f}", ha="center", fontsize=11)
ax.set_xticks(x, [NICE.get(m, m) + (" *" if m == final_model else "") for m in order], fontsize=12)
ax.set_ylim(0, 0.8); ax.set_ylabel("Balanced Accuracy", fontsize=13)
ax.set_title("Final Model Comparison \u2014 Spatial Cross-Validation", fontsize=15)
fig.text(0.5, -0.02, f"Score = mean across 5 outer spatial test folds ({primary} m blocks)"
         + (f"; error bars = SD" if SHOW_SD else "") + f"; * = final model (pre-declared rule); cells: {CELLS}",
         ha="center", fontsize=10)
save(fig, AGG, "figB_model_comparison")
display(s[["balanced_accuracy_mean", "balanced_accuracy_sd", "macro_f1_mean", "roc_auc_mean"]].round(4))

## Fig. C · Confusion matrix and mean spatial ROC curve of the final model

In [ ]:
f = main[main.model == final_model]
cm = confusion_matrix(f.y, f.y_pred, labels=[0, 1])
fig, axes = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1.1, 1]})
ax = axes[0]
im = ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]}", ha="center", va="center", fontsize=16, color="white" if cm[i, j] > (cm.min() + cm.max()) / 2 else "#17365C")
ax.set_xticks([0, 1], ["Sand", "Clay"], fontsize=13); ax.set_yticks([0, 1], ["Sand", "Clay"], fontsize=13)
ax.xaxis.tick_top(); ax.xaxis.set_label_position("top")
ax.set_xlabel("Prediction", fontsize=13); ax.set_ylabel("Actual", fontsize=13)
ax.set_title("Confusion Matrix", fontsize=20, pad=48)
for s_ in ax.spines.values():
    s_.set_visible(True)
fig.colorbar(im, ax=ax, fraction=0.046)
ax = axes[1]
grid = np.linspace(0, 1, 201); tprs, aucs = [], []
for k, g in f.groupby("outer_fold"):
    if g.y.nunique() < 2:
        continue
    fpr, tpr, _ = roc_curve(g.y, g.p_clay)
    tprs.append(np.interp(grid, fpr, tpr)); tprs[-1][0] = 0.0
    aucs.append(roc_auc_score(g.y, g.p_clay))
mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
ax.plot(grid, mean_tpr, color=NAVY, linewidth=2.2, label=f"Mean AUC = {np.mean(aucs):.4f}")
ax.plot([0, 1], [0, 1], linestyle="--", color="#777777")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_xlabel("False positive rate", fontsize=12); ax.set_ylabel("True positive rate", fontsize=12)
ax.set_title(f"{NICE.get(final_model, final_model)} \u2014 Mean Spatial ROC Curve ({primary} m blocks)", fontsize=12)
ax.legend(frameon=False, loc="upper left", fontsize=11)
fig.tight_layout()
save(fig, AGG, "figC_confusion_roc_final_model")
print("pooled outer test cells:", int(cm.sum()), "| fold AUCs:", np.round(aucs, 3))

## Fig. D · Explainable AI for the final model

The global SHAP importance is the mean |SHAP value| of each input on P(Clay). The held-out permutation importance is the drop in balanced accuracy on the outer test folds when that input is shuffled (notebook 07, mean over the 5 folds). The beeswarm shows one dot per explained held-out cell, coloured by the input's value.

In [ ]:
import shap
X07 = config.stage_dir(cfg, "07_xai", "aggregate")
sv = pd.read_csv(config.stage_dir(cfg, "07_xai", "restricted") / "shap_values_by_cell.csv")
sv = sv[sv.model == final_model]
if CELLS == "measured":
    sv = sv[sv.measured == 1]
feat = ["UTM_Easting_m", "UTM_Northing_m", "Top_Elevation_m"]
short = {"UTM_Easting_m": "Easting", "UTM_Northing_m": "Northing", "Top_Elevation_m": "Elevation"}
vals = sv[[f"shap_{short[c]}" for c in feat]].values
perm = pd.read_csv(X07 / "permutation_importance_summary.csv")
perm = perm[(perm.model == final_model) & (perm.kind == "single")
            & (perm.cells == ("all cells" if CELLS == "all" else "measured only"))]
perm = perm.set_index("input").reindex([short[c] for c in feat])
glob_ = pd.Series(np.abs(vals).mean(axis=0), index=feat).sort_values()
fig = plt.figure(figsize=(13, 9))
ax1 = fig.add_axes([0.08, 0.58, 0.38, 0.36]); ax2 = fig.add_axes([0.58, 0.58, 0.38, 0.36])
ax1.barh(glob_.index, glob_.values, color=MAGENTA)
ax1.set_xlabel("Mean absolute SHAP value"); ax1.set_title("Global SHAP importance", fontsize=11)
p = perm["mean"].rename(index={short[c]: c for c in feat}).sort_values()
ax2.barh(p.index, p.values, color=MAGENTA)
ax2.set_xlabel("Decrease in balanced accuracy"); ax2.set_title("Held-out permutation importance", fontsize=11)
ax3 = fig.add_axes([0.2, 0.06, 0.62, 0.42])
exp = shap.Explanation(values=vals, base_values=sv.base.values, data=sv[feat].values, feature_names=feat)
plt.sca(ax3)
shap.plots.beeswarm(exp, color=CMAP, show=False, plot_size=None)
ax3.set_title("SHAP beeswarm", fontsize=15)
save(fig, AGG, "figD_xai_final_model")
display(pd.DataFrame({"mean_abs_shap": pd.Series(np.abs(vals).mean(axis=0), index=feat),
                      "permutation_drop": perm["mean"].values}).round(4))

**Notes.** Fig. A shows borehole locations; keep it in `restricted/` until confidentiality is checked. In Fig. B the depth-only reference is shown in grey for comparison (it is not a candidate). Figs. B–D use the frozen final model and the outer-fold predictions; nothing is refitted.